# Practical excercise: BERT based persona judge

## Background

We are cooperating with NVIDIA and Valicon to produce a Slovene Nemotron Personas dataset. The dataset will contain synthetic Slovene persona description. It will be useful for various tasks such as market analysis and generating task-specific training data for models based on the interaction with a given persona.

In the first stage we generate demographic attributes based on the actual statistics for Slovenia. Next, we add the personality traits from the OCEAN Big Five specification. Based on these data, an LLM has to generate the general persona description. Specifically it has to generate the description for the following fields:
- `cultural_background`
- `skills_and_expertise`
- `career_goals_and_ambitions`
- `hobbies_and_interests`

However, the LLMs that generate these descriptions are prone to hallucinations and language errors. In this case, we denote with hallucinations part of the generated text that is inconsistent with the input data (wrong place to live, occupation, education level, etc.) or that is wrong in general (generating wrong factual information).

To prevent such errors, we implemented a LLM-as-a-judge approach, where LLM checks the generated personas and accepts or rejects them based on a certain criteria. To perform the scoring, the judge is first asked to find spans of text that include one of the following:
- **contradiction** — states something the record says is false (e.g. claims a job when the record says unemployed)
- **invention** — adds biographical detail that isn't in the record, but isn't contradicted by it either (allowed, even expected)
- **language** — a grammar, agreement, or wording error, unrelated to the record
- **correct** — a case handled well that could easily have gone wrong (e.g. correctly acknowledging retirement instead of inventing a job)
The judge determines the final quality of the generated persona based on the detected spans. If there are major contradictions, generation is rejected. If there are a lot of language errors, the generation is rejected. If the inventions are plaussible and consistent with the persona's bio, the generation gets higher score.

## Your task

You are given 200 Slovene persona generated with 3 different models: GaMS2-27B, Gemma3-27B and DeepSeek-V4-Flash and spans that were marked and labeled by Kimi-K3 judge. Your task is to find out whether a BERT model can be fine-tuned to correctly classify provided spans into the four classes described above.


## 1. Setup

Install `transformers`, `accelerate`, `datasets`, `evaluate`, `scikit-learn` and `pandas`.

In [1]:
!pip install -q transformers==4.57.6 accelerate datasets evaluate scikit-learn pandas

## 2. Get the data

As we work with the Colab, we first need to transfer the data to it. The data lives in the `exercise/data/` folder of the course GitHub repo: https://github.com/zivastebljaj/KCUI-efficient-adaptation-LLMs

Colab starts every session with an empty working directory (`/content`), so you need to bring the data in first. The repo is public, so the easiest way is to clone it straight from a code cell. Prefix a line with `!` to run it as a shell command:

```python
!git clone --depth 1 https://github.com/zivastebljaj/KCUI-efficient-adaptation-LLMs.git
DATA_DIR = "KCUI-efficient-adaptation-LLMs/exercise/data"
```

`--depth 1` downloads only the latest version, without the history. Afterwards, open the **Files** panel (folder icon in the left sidebar) and check that the data is there. You can also run `!ls -R {DATA_DIR}`. You should see:

```text
data/
├── generated_personas/
│   ├── personas_sl_deepseek.jsonl
│   ├── personas_sl_gams2.jsonl
│   └── personas_sl_gemma3.jsonl
└── judge_evaluations/
    ├── kimi_deepseek_analysis.jsonl
    ├── kimi_gams2_analysis.jsonl
    └── kimi_gemma3_analysis.jsonl
```

<details><summary>Alternative: download single files</summary>

Every file in the repo is also available as a raw download at
`https://raw.githubusercontent.com/zivastebljaj/KCUI-efficient-adaptation-LLMs/main/exercise/data/<path>`. Fetch it with `!wget -P <target_dir> <url>`, for example:

```python
!wget -q -P data/generated_personas https://raw.githubusercontent.com/zivastebljaj/KCUI-efficient-adaptation-LLMs/main/exercise/data/generated_personas/personas_sl_gams2.jsonl
```
</details>

<details><summary>Alternative: upload manually</summary>

On GitHub, click **Code → Download ZIP** and unzip it on your computer. Then, in Colab, drag the files from `exercise/data/` into the **Files** panel, or click its upload icon. Keep the folder structure shown above.
</details>

**Note:** Everything in `/content` is deleted when the Colab runtime disconnects or restarts. If that happens, run this step again.

In [2]:
!git clone --depth 1 https://github.com/zivastebljaj/KCUI-efficient-adaptation-LLMs.git
DATA_DIR = "KCUI-efficient-adaptation-LLMs/exercise/data"
!ls -R {DATA_DIR}

fatal: destination path 'KCUI-efficient-adaptation-LLMs' already exists and is not an empty directory.
KCUI-efficient-adaptation-LLMs/exercise/data:
generated_personas  judge_evaluations  processed_data

KCUI-efficient-adaptation-LLMs/exercise/data/generated_personas:
personas_sl_deepseek.jsonl  personas_sl_gams2.jsonl  personas_sl_gemma3.jsonl

KCUI-efficient-adaptation-LLMs/exercise/data/judge_evaluations:
kimi_deepseek_analysis.jsonl  kimi_gemma3_analysis.jsonl
kimi_gams2_analysis.jsonl

KCUI-efficient-adaptation-LLMs/exercise/data/processed_data:
test.jsonl  train.jsonl  val.jsonl


## 3. Build the dataset

When dealing with real-world problems, the data can be all over the place. In this case, the data is split across three generation files and three judge files. Your first task is to join them all together in a single file and split it into train, validation and test sets.

**Note:** If you have troubles with this task or simply want to skip it, we added built dataset to the repository in `exercise/data/processed_data` folder. You can skip this task and go straight to task 4. However, we stronlgy encourage you to try this task to get familiar with the data.

### The files

All files are JSON Lines (one JSON object per line).

Generation files (`personas_sl_{generator}.jsonl`) hold one generation per person, identified by `(uuid, generation_index)`. When we prepared the dataset for you, we reduced the multiple generations per person to a single generation. Hence, you can uniquely identify the person by using `uuid` only (`generation_index` is used when multiple generations per person are present).

The fields you'll need:

| field | meaning |
|---|---|
| `first_name`, `last_name`, `sex`, `age` | basic demographics |
| `marital_status`, `household_status` | family situation |
| `education_level`, `bachelors_field` | education (`bachelors_field` contains `"brez diplome"` if there is no degree) |
| `activity_status` | work status (employed, retired, unemployed, …) |
| `occupation`, `detailed_occupation` | job; may be empty |
| `statistical_region`, `municipality`, `post_code`, `post_town`, `settlement` | location |
| `openness`, `conscientiousness`, `extraversion`, `agreeableness`, `neuroticism` | Big Five traits, each a dict with a `label` key |

Judge files (`judge_output/kimi_{generator}_analysis.jsonl`) hold one verdict per generation, with `uuid`, `generation_index` and a list of `spans`. Each span has `text`, `block`, `mark` (this is the label), `covers` and `field`. You will need at least `text` (for the input) and `mark` fields.

### 3.1 Build a single dataset

You first need to join all six files together in a dataset, that can be used for training. Your goal should be a dataset with **one row per flagged span**, containing the span text, the person's record (it will be later used as context), and the mark (span label). Your tasks are:
- match the generated personas with judge verdicts
- join the data from all three generators together
- create multiple rows from the matched records, one for each text span from the judge file

Rules:
- skip spans that stand for a whole bullet list (`covers == "bullet_list"`), since those are list fragments rather than sentences
- attach the same persona attributes to every span corresponding to the same persona, regardless of its mark
- drop duplicate spans

Check what you built. How many rows do you have, and how are the labels distributed? Read a few rows and make sure the examples make sense.

### 3.2 Train, validation, test split

Split the built dataset from the previous subtask into train, validation and test split. The subset ratios should be:
- Train set - 80 %
- Validation set - 10 %
- Test set - 10 %
Validation set will be used to monitor the overfitting during the training and the test set will be used for the final evaluation of the models.

**Important:** Make sure that no person is not in multiple splits. If example with `uuid` equal to `x` lands in the train set, all examples with `uuid` equal to `x` should be in the train set. The same rule applies for validation and test set.


In [3]:
import json
import pandas as pd

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

# 3.1 Build a single dataset
rows = []
for gen in ["deepseek", "gams2", "gemma3"]:
    personas = read_jsonl(f"{DATA_DIR}/generated_personas/personas_sl_{gen}.jsonl")
    personas = {p["uuid"]: p for p in personas}
    judge = read_jsonl(f"{DATA_DIR}/judge_evaluations/kimi_{gen}_analysis.jsonl")

    for j in judge:
        p = personas[j["uuid"]]
        for span in j["spans"]:
            if span["covers"] == "bullet_list":
                continue
            row = {"uuid": j["uuid"], "generator": gen, "text": span["text"], "label": span["mark"]}
            for key in ["first_name", "last_name", "sex", "age", "marital_status", "household_status",
                        "education_level", "bachelors_field", "activity_status", "occupation",
                        "detailed_occupation", "statistical_region", "municipality", "post_code",
                        "post_town", "settlement"]:
                row[key] = p.get(key)
            for trait in ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]:
                row[trait] = p[trait]["label"]
            rows.append(row)

df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=["uuid", "generator", "text"])
print(len(df), "rows")
print(df["label"].value_counts())
df.sample(5, random_state=0)

4058 rows
label
invention        2631
correct           817
language          555
contradiction      55
Name: count, dtype: int64


,uuid,generator,text,label,first_name,last_name,sex,age,marital_status,household_status,...,statistical_region,municipality,post_code,post_town,settlement,openness,conscientiousness,extraversion,agreeableness,neuroticism
914,b06e223b-9e13-48c2-82e5-5cc6d8735583,deepseek,Njena vizija je ustanoviti lokalno umetnostno-...,invention,Cecilija,Alibabić,Ženska,77,Poročeni,Zakonec brez otrok,...,Gorenjska,Kranj,4000,Kranj,Britof,very high,average,high,average,high
3122,5cf561b1-f4c0-49e4-ab6d-0d8f1eed9428,gemma3,"predvsem Pohorje, ki ga obdaja",invention,Ivan,Bračun,Moški,29,Samski,Otrok v enostarševski družini živi z očetom,...,Koroška,Dravograd,2373,Šentjanž pri Dravogradu,Otiški Vrh,high,average,average,average,high
1900,62309994-428e-4452-b79e-595d532f0e32,gams2,"Zanima se za ročna dela, predvsem za pletenje ...",invention,Majda,Mršić,Ženska,62,Poročeni,Zakonec z otroki,...,Savinjska,Velenje,3320,Velenje,Škale,average,high,high,low,low
3071,4f797db9-56c9-4ec2-b6af-fb37cdff663e,gemma3,da se njeno delo cenjeno,language,Rozalija,Perhavec,Ženska,63,Samski,Mati z otroki,...,Obalno-kraška,Koper,6000,Koper - Capodistria,Bošamarin,average,average,high,high,average
572,66bc041f-72e4-4f22-b5a4-63a5608339f8,deepseek,"Although Marija is approaching retirement, she...",language,Marija,Albreht,Ženska,65,Poročeni,Zakonec brez otrok,...,Gorenjska,Jesenice,4270,Jesenice,Jesenice,average,low,high,low,very low


In [4]:
# 3.2 Train / validation / test split (80/10/10), grouped by uuid
import numpy as np

uuids = df["uuid"].unique()
rng = np.random.default_rng(42)
rng.shuffle(uuids)

n = len(uuids)
train_uuids = uuids[:int(0.8 * n)]
val_uuids = uuids[int(0.8 * n):int(0.9 * n)]
test_uuids = uuids[int(0.9 * n):]

train_df = df[df["uuid"].isin(train_uuids)]
val_df = df[df["uuid"].isin(val_uuids)]
test_df = df[df["uuid"].isin(test_uuids)]
print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

train: 3271 val: 372 test: 415


## 4. Prepare data for BERT Training

### 4.1 Convert the data into correct format

Currently, we have the data in format that is appropriate for classic machine learning models - a set of features and a label. However, BERT models work with text. Therefore, your next task will be to turn the current data into a textual input. On the other hand, labels such as "contradiction", "language", etc. do not tell BERT models anything.

Specific instructions:
- Turn the persona information into a textual input for the BERT models (**you do not need a prompt, just a textual input**)
- Keep the text span as a separate field (you will join it together with the persona info later)
- Drop the classes with fewer than 5 examples (if  they exist)
- Convert the labels to integer IDs

**Important:** Don't use the judge's annotations as context — the input context should be based only on persona's attributes

<details><summary>Hint: the context string</summary>

Keep it simple and readable, for example:
`Ime: Nada Hren | Spol: ženski | Starost: 67 | ... | Poklic: ... | OCEAN: openness=visoko, ...`

Leave out empty fields like a missing occupation, rather than writing `None`.
</details>

#### Bonus task

Try different text preparation strategies and observe how they impact the model's performance. Maybe you do not need the whole context for every label (hint: language label might not need the context at all).

### 4.2 Merge the context and the input

Put together context and the input and tokenize it, so you get the actual input for the BERT model.

<details><summary>Hint: separation token</summary>

BERT tokenizer includes`[SEP]` token that can serve as a separator.
</details>

In [6]:
# 4.1 Persona -> text, labels -> ids
def make_context(r):
    parts = [
        f"Ime: {r['first_name']} {r['last_name']}",
        f"Spol: {r['sex']}",
        f"Starost: {r['age']}",
        f"Zakonski stan: {r['marital_status']}",
        f"Gospodinjstvo: {r['household_status']}",
        f"Izobrazba: {r['education_level']}",
    ]
    if r["bachelors_field"] and "brez diplome" not in r["bachelors_field"]:
        parts.append(f"Podrocje: {r['bachelors_field']}")
    parts.append(f"Delovni status: {r['activity_status']}")
    if r["occupation"]:
        parts.append(f"Poklic: {r['occupation']}")
    if r["detailed_occupation"]:
        parts.append(f"Podrobni poklic: {r['detailed_occupation']}")
    parts.append(f"Kraj: {r['settlement']}, {r['municipality']}, {r['statistical_region']}")
    parts.append(f"OCEAN: openness={r['openness']}, conscientiousness={r['conscientiousness']}, "
                 f"extraversion={r['extraversion']}, agreeableness={r['agreeableness']}, "
                 f"neuroticism={r['neuroticism']}")
    return " | ".join(parts)

train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()

label_list = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print(label2id)

for d in [train_df, val_df, test_df]:
    d["context"] = d.apply(make_context, axis=1)
    d["labels"] = d["label"].map(label2id)

print(train_df["context"].iloc[0])

{'contradiction': 0, 'correct': 1, 'invention': 2, 'language': 3}
Ime: Jelena Uršič | Spol: Ženska | Starost: 38 | Zakonski stan: Poročeni | Gospodinjstvo: Zakonec z otroki | Izobrazba: Visokošolska 1. stopnje ipd. | Podrocje: Naravoslovje, matematika in statistika | Delovni status: Zaposleni | Poklic: Tehniki in drugi strokovni sodelavci | Podrobni poklic: 3313 Knjigovodje in strokovni sodelavci v računovodstvu | Kraj: Dolenje, Ajdovščina, Goriška | OCEAN: openness=average, conscientiousness=average, extraversion=average, agreeableness=average, neuroticism=very high


In [7]:
# 4.2 Merge context + span and tokenize
from transformers import AutoTokenizer

model_name = "EMBEDDIA/sloberta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("SEP token:", tokenizer.sep_token)

MAX_LENGTH = 256

def tokenize_example(example):
    text = example["context"] + f" {tokenizer.sep_token} " + example["text"]
    tokenized = tokenizer(text, truncation=True, max_length=MAX_LENGTH)
    tokenized["labels"] = int(example["labels"])
    return tokenized

train_ds = [tokenize_example(example) for _, example in train_df.iterrows()]
val_ds = [tokenize_example(example) for _, example in val_df.iterrows()]
test_ds = [tokenize_example(example) for _, example in test_df.iterrows()]
print("First training example:", tokenizer.decode(train_ds[0]["input_ids"]))

SEP token: </s>
First training example: <s> Ime: Jelena Uršič | Spol: Ženska | Starost: 38 | Zakonski stan: Poročeni | Gospodinjstvo: Zakonec z otroki | Izobrazba: Visokošolska 1. stopnje ipd. | Podrocje: Naravoslovje, matematika in statistika | Delovni status: Zaposleni | Poklic: Tehniki in drugi strokovni sodelavci | Podrobni poklic: 3313 Knjigovodje in strokovni sodelavci v računovodstvu | Kraj: Dolenje, Ajdovščina, Goriška | OCEAN: openness=average, conscientiousness=average, extraversion=average, agreeableness=average, neuroticism=very high </s> Odraščala je ob tradicionalnih primorskih običajih, kot so priprava lokalnih jedi in praznovanje krajevnih praznikov.</s>


## 5. Train

Finetune `EMBEDDIA/sloberta` for sequence classification. Prepare the training hyperparameters as we did in demo and try to run the training.

Try different hyperparameters and check how they affect the training performance. Track accuracy and macro-F1.

<details><summary>Hint: the pieces you need</summary>

`AutoTokenizer`, `AutoModelForSequenceClassification`, `datasets.Dataset.from_dict`, `DataCollatorWithPadding`, `TrainingArguments`, `Trainer`. Name the label column `labels`, since that's what the Trainer expects.
</details>

### Bonus Task

Try different classification settings (for example trying to classify only language) and observe how the model's performance changes.


In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(label_list), id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir="./persona_judge",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at EMBEDDIA/sloberta and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.782027,0.733871,0.426657
2,No log,0.646976,0.779570,0.502664
3,0.766800,0.586191,0.803763,0.538418


TrainOutput(global_step=615, training_loss=0.7222009426209984, metrics={'train_runtime': 319.0059, 'train_samples_per_second': 30.761, 'train_steps_per_second': 1.928, 'total_flos': 885357877326624.0, 'train_loss': 0.7222009426209984, 'epoch': 3.0})

## 6. Evaluate

Print a per-class classification report and a confusion matrix for the test set. Which marks are easy, and which get confused with each other?

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred, target_names=label_list, digits=3))
print("Confusion matrix (rows = true, cols = predicted):", label_list)
print(confusion_matrix(y_true, y_pred))

               precision    recall  f1-score   support

contradiction      0.000     0.000     0.000         3
      correct      0.614     0.531     0.570        81
    invention      0.797     0.931     0.859       262
     language      0.872     0.493     0.630        69

     accuracy                          0.773       415
    macro avg      0.571     0.489     0.515       415
 weighted avg      0.768     0.773     0.758       415

Confusion matrix (rows = true, cols = predicted): ['contradiction', 'correct', 'invention', 'language']
[[  0   3   0   0]
 [  0  43  36   2]
 [  0  15 244   3]
 [  0   9  26  34]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
